# 05. One-stage anchor-free: FCOS

1. **Location assignment without anchors:** how many feature-map locations each object gets, by size.
2. **Sweep on fold 0:** the default model, two centre-sampling radii and a box-loss modification.
3. **Full cross-validation** of the best experiment; comparison with the tuned anchor-based detectors by object size.

## Setup

In [ ]:
import subprocess
import sys
from pathlib import Path

if Path("../src/core").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/tmp/repo")  # outside /kaggle/working so it is not saved as output
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

In [ ]:
!pip install -q faster-coco-eval

In [ ]:
import pandas as pd

from core import PROTOCOL
from core.cv import compare, cv_table, environment, run_cv
from core.data import assign_folds, find_data_root, load_pool, pool_targets
from core.fusion import fold_ensemble
from core.submission import write_submission
from core.tv import fcos_assignment_stats, make_tv_fns, plot_histories

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")
WORK.mkdir(parents=True, exist_ok=True)
environment()

## Data and folds

In [ ]:
DATA = find_data_root()
frames, gt = load_pool(DATA)
targets = pool_targets(gt)
frames["fold"] = frames["video"].map(assign_folds(frames))
frames.groupby("fold").agg(clips=("video", "nunique"), frames=("uid", "size"))

## How FCOS assigns locations to objects
There are no anchors. Every location of the feature maps (strides 8–128) predicts a class and the distances
to the four box sides. A location is a positive for an object if it lies inside the box **and** within
`radius × stride` of the box centre (centre sampling); each object is handled by the pyramid level that
matches its size, and a location inside several boxes takes the smallest one.

- **Small objects:** an object of ~10 px covers about one stride-8 cell, so a small radius can leave it with no
  positive location at all; a larger radius gives it more training signal.
- **Dense scenes:** in crowds the sampling regions of neighbours overlap. A location is given to one box only,
  so a larger radius adds ambiguous positives that lie on a neighbouring object.

Positive locations at the finest level per training box (training folds of fold 0), for the searched radii:

In [ ]:
RADII = [0.5, 1.5, 2.5]  # search range; 1.5 is the torchvision default
train0 = frames[frames["fold"] != 0]
pd.concat({f"radius {r}": fcos_assignment_stats(train0, targets, PROTOCOL["imgsz"], r) for r in RADII}, axis=1)

## Training setup
FCOS ResNet-50-FPN, COCO-pretrained, new classification layer; long side 1280 px, every 5th frame,
ignored regions masked, horizontal flip; SGD, lr 0.0025, warm-up + cosine, 8 epochs, last epoch kept.

Experiments, one change each:
- **Centre-sampling radius 0.5 and 2.5** (default 1.5): the assignment search range above.
- **Box-regression loss ×2** (loss modification): more weight on localisation.

In [ ]:
BASE = dict(arch="fcos", lr=0.0025)
EXPERIMENTS = {
    "fcos_default": BASE,
    "fcos_radius0.5": {**BASE, "center_sampling_radius": 0.5},
    "fcos_radius2.5": {**BASE, "center_sampling_radius": 2.5},
    "fcos_boxloss2": {**BASE, "loss_weights": {"bbox_regression": 2.0}},
}
ANCHOR_BASED = ["retina_default", "retina_anchors", "retina_anchors_iou", "retina_no_focal",
                "frcnn_default", "frcnn_roi_pos50", "frcnn_anchors", "frcnn_boxloss2", "frcnn_step_lr"]  # 03, 04


def config(name):
    return {**PROTOCOL, **EXPERIMENTS[name], "name": name}

## Sweep on fold 0

In [ ]:
for name in EXPERIMENTS:
    print(name, EXPERIMENTS[name])
    run_cv(frames, gt, config(name), *make_tv_fns(config(name), WORK, gt, targets), WORK, folds=[0])

### Fold-0 comparison by object size
Initial vs tuned FCOS, the anchor-based detectors and the YOLO baseline on the same held-out fold.

In [ ]:
sweep = compare([*EXPERIMENTS, *ANCHOR_BASED, "baseline"], WORK, fold=0)
display(sweep.round(4))
plot_histories(EXPERIMENTS, 0, WORK);

## Full cross-validation of the best experiment
Chosen on fold 0, so its fold-0 score is slightly optimistic.

In [ ]:
BEST = sweep.loc[sweep.index.isin(EXPERIMENTS), "mAP"].idxmax()
print("best on fold 0:", BEST)
result = run_cv(frames, gt, config(BEST), *make_tv_fns(config(BEST), WORK, gt, targets), WORK)

In [ ]:
display(compare([BEST, *ANCHOR_BASED, "baseline"], WORK).round(4))  # out-of-fold, by object size
cv_table(result).round(4)

### Findings
*(After the run: selected radius and why, AP_small / AP_medium / AP_large of FCOS vs the tuned anchor-based
detectors, and whether the assignment table explains the difference.)*

## Submission
The four fold models of the best experiment, fused by Weighted Boxes Fusion.

In [ ]:
test = pd.read_csv(DATA / "test_images.csv")
_, predict_fn = make_tv_fns(config(BEST), WORK, gt, targets)
test_dets = fold_ensemble(BEST, predict_fn, [str(DATA / f) for f in test["file_name"]], WORK)
write_submission(test_dets, test, WORK / f"submission_{BEST}.csv").head()